# Bootstrap particle filter lab

Track a scalar latent state with a bootstrap particle filter, inspect posterior moments and ESS, then use FFBSi to sample complete smoothed trajectories with future observations informing earlier states.

In [ ]:
import sys
from pathlib import Path
repo_root = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
                  if (candidate / "implementations/python/state-estimation/particle_filter.py").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repository root for state-estimation examples")
sys.path.insert(0, str(repo_root / "implementations/python/state-estimation"))
import numpy as np
from particle_filter import bootstrap_particle_filter
from particle_smoother import ffbsi_particle_smoother
rng = np.random.default_rng(3)
truth = 0.0
observations = []
for _ in range(40):
    truth = 0.85 * truth + rng.normal(0, 0.25)
    observations.append(truth + rng.normal(0, 0.35))


In [ ]:
initial = np.random.default_rng(10).normal(0, 1, size=(4000, 1))
transition = lambda p, gen: 0.85 * p + gen.normal(0, 0.25, size=p.shape)
log_likelihood = lambda p, z: -0.5 * ((z - p[0]) / 0.35) ** 2
fit = bootstrap_particle_filter(observations, initial, transition, log_likelihood, rng=np.random.default_rng(11))
print('final filtered mean:', fit['means'][-1])
print('final effective sample size:', fit['effective_sample_sizes'][-1])
print('resampling count:', fit['resampled'].sum())
transition_log_density = lambda previous, following: -0.5 * ((following[0] - 0.85 * previous[0]) / 0.25) ** 2
smoothed = ffbsi_particle_smoother(fit, transition_log_density, n_trajectories=200, rng=np.random.default_rng(12))
print('smoothed means, first five:', smoothed['means'][:5, 0])
print('trajectory shape:', smoothed['trajectories'].shape)


## Interpretation
The posterior is represented by weighted particles, not only a mean and covariance. A low effective sample size indicates weight concentration and triggers resampling. FFBSi uses the stored filtering distributions and the transition density to sample backward from the final time, so future observations can revise earlier state estimates. It is offline and depends on a transition log density consistent with the forward model; it is not simply a reverse call to the transition sampler.